In [ ]:
!pip install -q streamlit PyPDF2 google-genai
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1



In [ ]:
%%writefile app.py

import streamlit as st
import PyPDF2
import io
from google import genai

GEMINI_API_KEY = "AQ.Ab8RN6LdGmbFTI4caZxUWoYEneK-pMo5NqNHIW_qKA000djGsQ"

client = genai.Client(api_key=GEMINI_API_KEY)

st.set_page_config(
    page_title="AI Resume Critiquer",
    page_icon="📄",
    layout="centered"
)

st.title("AI Resume Critiquer")
st.write("Upload your resume and get AI-powered feedback.")

resume = st.file_uploader(
    "Upload your resume",
    type=["pdf", "txt"]
)

job_role = st.text_input(
    "What job are you applying for?",
    placeholder="Example: Data Scientist"
)


def get_pdf_text(file):
    reader = PyPDF2.PdfReader(file)
    text = ""

    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text + "\n"

    return text


def get_file_text(file):
    if file.type == "application/pdf":
        return get_pdf_text(io.BytesIO(file.read()))

    return file.read().decode("utf-8")


if st.button("Analyze Resume"):

    if resume is None:
        st.warning("Please upload your resume first.")
        st.stop()

    try:
        resume_text = get_file_text(resume)
    except Exception as e:
        st.error(f"Could not read the file: {e}")
        st.stop()

    if not resume_text.strip():
        st.error("No readable text was found in the file.")
        st.stop()

    role = job_role.strip() or "general job applications"

    prompt = f"""
Act as a professional recruiter and resume reviewer.

The candidate is applying for:
{role}

Here is the resume:

{resume_text}

Review the resume and give practical feedback.

Include:

1. Overall score out of 10
2. Main strengths
3. Main weaknesses
4. Skills analysis
5. Work experience analysis
6. Project analysis
7. Education analysis
8. ATS compatibility
9. Missing or useful keywords for the target role
10. Grammar and wording problems
11. Specific improvements

For weak sentences, show a better version.

Keep the feedback clear, practical and easy to understand.
"""

    try:
        with st.spinner("Analyzing your resume..."):

            response = client.models.generate_content(
                model="gemini-2.5-flash",
                contents=prompt
            )

        st.subheader("Analysis Results")

        if response.text:
            st.write(response.text)
        else:
            st.error("Gemini did not return a response.")

    except Exception as e:
        st.error(f"Something went wrong: {e}")

Writing app.py


In [ ]:
!pkill -f streamlit || true
!pkill -f cloudflared || true
!pkill -f uvicorn || true
!streamlit run app.py --server.address 0.0.0.0 --server.port 8501 > /content/streamlit.log 2>&1 &

^C
^C
^C


In [22]:
!sleep 3
!cloudflared tunnel --url http://127.0.0.1:8501

2026-09-14T05:49:41Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-09-14T05:49:41Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-09-14T05:49:44Z INF +--------------------------------------------------------------------------------------------+
2026-09-14T05:49:44Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-09-14T05:49:44Z INF |  https://associates-interviews-institute-dev.trycloudf